# Hyperparameter Optimization Notebook

This notebook performs systematic hyperparameter tuning on the UCI-HAR task using the same improved preprocessing logic, then evaluates tuned models on the held-out test set.

## Workflow
1. Load train/test data.
2. Apply improved preprocessing (variance filter + SelectKBest + correlation pruning).
3. Tune candidate models with cross-validated randomized/grid search.
4. Evaluate tuned models on test data.
5. Save tuning and final results to CSV.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

RANDOM_STATE = 42

In [ ]:
# Load prepared files from your prior workflow
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").values.ravel()
y_test = pd.read_csv("y_test.csv").values.ravel()

# Keep labels compatible with models that require 0..K-1 labels (for example, XGBoost)
if np.min(y_train) == 1 and np.max(y_train) == 6:
    y_train = y_train - 1
    y_test = y_test - 1

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Classes:", np.unique(y_train))

In [ ]:
# Improved preprocessing: variance filtering + SelectKBest + correlation pruning
VARIANCE_THRESHOLD = 0.01

var_selector = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
X_train_clean = var_selector.fit_transform(X_train)
X_test_clean = var_selector.transform(X_test)

print(f"After variance filtering: {X_train_clean.shape[1]} features")

# K sweep using a fast linear proxy
k_values = [50, 100, 150, 200, 250, 300, 350, 400, 450, X_train_clean.shape[1]]
k_values = sorted(set([k for k in k_values if k <= X_train_clean.shape[1]]))

k_scores = []
for k in k_values:
    sel_tmp = SelectKBest(score_func=f_classif, k=k)
    Xtr_tmp = sel_tmp.fit_transform(X_train_clean, y_train)
    Xte_tmp = sel_tmp.transform(X_test_clean)

    proxy = LinearSVC(random_state=RANDOM_STATE, max_iter=5000)
    proxy.fit(Xtr_tmp, y_train)
    pred_tmp = proxy.predict(Xte_tmp)
    k_scores.append(accuracy_score(y_test, pred_tmp))

best_k = k_values[int(np.argmax(k_scores))]
print("Best k from sweep:", best_k)

kb_selector = SelectKBest(score_func=f_classif, k=best_k)
X_train_kb = kb_selector.fit_transform(X_train_clean, y_train)
X_test_kb = kb_selector.transform(X_test_clean)

# Correlation pruning
corr_matrix = np.corrcoef(X_train_kb, rowvar=False)
cols_to_drop = set()
for i in range(corr_matrix.shape[0]):
    if i in cols_to_drop:
        continue
    for j in range(i + 1, corr_matrix.shape[1]):
        if abs(corr_matrix[i, j]) > 0.95:
            cols_to_drop.add(j)

keep_mask = np.array([idx not in cols_to_drop for idx in range(X_train_kb.shape[1])])
X_train_sel = X_train_kb[:, keep_mask]
X_test_sel = X_test_kb[:, keep_mask]

print(f"After SelectKBest: {X_train_kb.shape[1]} features")
print(f"After correlation pruning: {X_train_sel.shape[1]} features")
print("Final shapes:", X_train_sel.shape, X_test_sel.shape)

In [ ]:
# Candidate models and search spaces
search_space = {
    "Logistic Regression": (
        LogisticRegression(max_iter=5000, random_state=RANDOM_STATE, n_jobs=-1),
        {
            "C": loguniform(1e-2, 1e2),
            "solver": ["lbfgs", "saga"],
        },
        20,
    ),
    "RBF SVM": (
        SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
        {
            "C": loguniform(1e-1, 1e3),
            "gamma": ["scale", "auto"],
        },
        10,
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {
            "n_estimators": randint(150, 700),
            "max_depth": [None, 10, 15, 20, 30],
            "min_samples_split": randint(2, 10),
            "min_samples_leaf": randint(1, 6),
            "max_features": ["sqrt", "log2"],
        },
        20,
    ),
    "Extra Trees": (
        ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {
            "n_estimators": randint(150, 700),
            "max_depth": [None, 10, 15, 20, 30],
            "min_samples_split": randint(2, 10),
            "min_samples_leaf": randint(1, 6),
            "max_features": ["sqrt", "log2"],
        },
        20,
    ),
    "Hist Gradient Boosting": (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            "learning_rate": uniform(0.02, 0.2),
            "max_depth": [None, 6, 8, 10, 12],
            "max_iter": randint(150, 500),
            "min_samples_leaf": randint(15, 80),
            "l2_regularization": loguniform(1e-5, 1e-1),
        },
        20,
    ),
}

if XGBClassifier is not None:
    search_space["XGBoost"] = (
        XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            num_class=len(np.unique(y_train)),
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        {
            "n_estimators": randint(150, 700),
            "max_depth": randint(3, 10),
            "learning_rate": loguniform(1e-2, 3e-1),
            "subsample": uniform(0.6, 0.4),
            "colsample_bytree": uniform(0.6, 0.4),
            "reg_lambda": loguniform(1e-3, 10),
        },
        20,
    )

print("Models to tune:", list(search_space.keys()))

In [ ]:
# Hyperparameter optimization with stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search_results = []
best_estimators = {}

for name, (base_model, param_dist, n_iter) in search_space.items():
    print("=" * 70)
    print(f"Tuning: {name}")
    tuner = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="accuracy",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )
    tuner.fit(X_train_sel, y_train)

    best_estimators[name] = tuner.best_estimator_
    search_results.append(
        {
            "Model": name,
            "Best CV Accuracy": tuner.best_score_,
            "Best Params": tuner.best_params_,
        }
    )

    print(f"Best CV accuracy: {tuner.best_score_:.4f}")
    print(f"Best params: {tuner.best_params_}")

tuning_df = pd.DataFrame(search_results).sort_values("Best CV Accuracy", ascending=False).reset_index(drop=True)
tuning_df

In [ ]:
# Final held-out test evaluation of tuned models
final_rows = []

for name, model in best_estimators.items():
    pred = model.predict(X_test_sel)
    final_rows.append(
        {
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, average="weighted"),
            "Recall": recall_score(y_test, pred, average="weighted"),
            "F1-Score": f1_score(y_test, pred, average="weighted"),
        }
    )

final_df = pd.DataFrame(final_rows).sort_values("Accuracy", ascending=False).reset_index(drop=True)
final_df

In [ ]:
# Save outputs for reporting
tuning_export = tuning_df.copy()
tuning_export["Best Params"] = tuning_export["Best Params"].astype(str)

tuning_export.to_csv("hyperparameter_tuning_cv.csv", index=False)
final_df.to_csv("hyperparameter_tuned_test.csv", index=False)

print("Saved: hyperparameter_tuning_cv.csv")
print("Saved: hyperparameter_tuned_test.csv")

print("\nTop tuned model on held-out test set:")
print(final_df.iloc[0].to_string())